# AI-Powered Personal Spending & Financial Behavior Coach
## Notebook 01 — Data Acquisition & Wrangling

### Objective
Prepare raw transaction data for downstream analytics, clustering, and LLM-generated coaching insights.

This notebook:
1. Loads the raw transaction dataset
2. Standardizes schema (column names, types, formats)
3. Validates data quality (missingness, duplicates, semantic checks)
4. Engineers foundational features (time features, age features)
5. Persists cleaned, analysis-ready datasets

In [27]:
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

In [29]:
DATA_PATH = "credit_card_transaction_flow.csv"
df_raw = pd.read_csv(DATA_PATH)

df_raw.head()

,Customer ID,Name,Surname,Gender,Birthdate,Transaction Amount,Date,Merchant Name,Category
0,752858,Sean,Rodriguez,F,20-10-2002,35.47,03-04-2023,Smith-Russell,Cosmetic
1,26381,Michelle,Phelps,NaN,24-10-1985,2552.72,17-07-2023,"Peck, Spence and Young",Travel
2,305449,Jacob,Williams,M,25-10-1981,115.97,20-09-2023,Steele Inc,Clothing
3,988259,Nathan,Snyder,M,26-10-1977,11.31,11-01-2023,"Wilson, Wilson and Russell",Cosmetic
4,764762,Crystal,Knapp,F,02-11-1951,62.21,13-06-2023,Palmer-Hinton,Electronics


## 1) Initial Data Profiling

We start by reviewing:
- shape and column types
- missing values
- basic descriptive statistics

This helps determine cleaning strategy and data quality rules.

In [32]:
print("Shape:", df_raw.shape)
df_raw.info()

missing = df_raw.isna().sum().sort_values(ascending=False)
missing_pct = (df_raw.isna().mean() * 100).round(2).sort_values(ascending=False)

pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})

Shape: (50000, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Customer ID         50000 non-null  int64  
 1   Name                50000 non-null  object 
 2   Surname             50000 non-null  object 
 3   Gender              44953 non-null  object 
 4   Birthdate           50000 non-null  object 
 5   Transaction Amount  50000 non-null  float64
 6   Date                50000 non-null  object 
 7   Merchant Name       50000 non-null  object 
 8   Category            50000 non-null  object 
dtypes: float64(1), int64(1), object(7)
memory usage: 3.4+ MB


,missing_count,missing_pct
Gender,5047,10.09
Customer ID,0,0.00
Name,0,0.00
Surname,0,0.00
Birthdate,0,0.00
Transaction Amount,0,0.00
Date,0,0.00
Merchant Name,0,0.00
Category,0,0.00


## 2) Standardize Column Names

To support reproducibility and consistent downstream processing, we:
- convert to lowercase
- replace spaces with underscores

In [35]:
df = df_raw.copy()

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df.columns

Index(['customer_id', 'name', 'surname', 'gender', 'birthdate', 'transaction_amount', 'date', 'merchant_name',
       'category'],
      dtype='object')

## 3) Validate Expected Columns

We validate the schema to catch upstream changes early.
If any required fields are missing, we should stop and investigate.

In [38]:
required_cols = {
    "customer_id",
    "name",
    "surname",
    "gender",
    "birthdate",
    "transaction_amount",
    "date",
    "merchant_name",
    "category"
}

missing_required = required_cols - set(df.columns)
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

print("✅ Required columns present.")

✅ Required columns present.


## 4) Parse Dates (Robust Handling for DD-MM-YYYY / Mixed Formats)

The dataset contains date strings like "17-07-2023" (DD-MM-YYYY), which is not the default month-first format.

We use Pandas mixed-format parsing with `dayfirst=True` and `errors='coerce'`:
- Handles mixed formats safely
- Converts invalid formats to NaT (so we can audit)

In [41]:
# Parse date columns robustly
df["date"] = pd.to_datetime(df["date"], format="mixed", dayfirst=True, errors="coerce")
df["birthdate"] = pd.to_datetime(df["birthdate"], format="mixed", dayfirst=True, errors="coerce")

# Validate parse success
date_null_rate = df[["date", "birthdate"]].isna().mean().mul(100).round(3)
date_null_rate

date         0.0
birthdate    0.0
dtype: float64

In [43]:
failed_dates = df[df["date"].isna() | df["birthdate"].isna()][["customer_id", "date", "birthdate"]]
print("Failed date rows:", len(failed_dates))
failed_dates.head(10)

Failed date rows: 0


,customer_id,date,birthdate


## 5) Handle Missing Values

We avoid dropping rows unless the field is critical.

Observed missingness:
- `gender` has missing values (~10%)

Strategy:
- Fill missing gender with "Unknown"
- Keep the records to preserve behavioral patterns

In [46]:
df["gender"] = df["gender"].fillna("Unknown")
df["gender"].value_counts(dropna=False).head(10)

gender
F          22713
M          22240
Unknown     5047
Name: count, dtype: int64

## 6) Duplicates & Integrity Checks

We check for:
- duplicated rows (exact duplicates)
- suspicious values (e.g., zero or negative amounts)
- missing key fields

In [49]:
dup_rows = df.duplicated().sum()
print("Duplicate rows:", dup_rows)

# If duplicates exist, remove exact duplicates
if dup_rows > 0:
    df = df.drop_duplicates().copy()
    print("✅ Dropped duplicate rows. New shape:", df.shape)

Duplicate rows: 0


In [51]:
df["transaction_amount"].describe()

count    50000.000000
mean       442.119239
std        631.669724
min          5.010000
25%         79.007500
50%        182.195000
75%        470.515000
max       2999.880000
Name: transaction_amount, dtype: float64

In [53]:
zero_amt = (df["transaction_amount"] == 0).sum()
print("Zero-amount transactions:", zero_amt)

df = df[df["transaction_amount"] != 0].copy()
print("Shape after removing zeros:", df.shape)

Zero-amount transactions: 0
Shape after removing zeros: (50000, 9)


## 7) Text Cleaning (Merchant Name)

Merchant names often include punctuation and inconsistent casing.
We create a normalized text field that supports:
- merchant pattern analysis
- future NLP classification models
- stable grouping

In [56]:
def clean_text(s: str) -> str:
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9\s]", " ", s)     # keep letters/numbers/spaces
    s = re.sub(r"\s+", " ", s).strip()    # collapse whitespace
    return s

df["merchant_name_clean"] = df["merchant_name"].apply(clean_text)

df[["merchant_name", "merchant_name_clean"]].head(10)

,merchant_name,merchant_name_clean
0,Smith-Russell,smith russell
1,"Peck, Spence and Young",peck spence and young
2,Steele Inc,steele inc
3,"Wilson, Wilson and Russell",wilson wilson and russell
4,Palmer-Hinton,palmer hinton
5,"Tran, Torres and Joyce",tran torres and joyce
6,"Evans, Griffin and Torres",evans griffin and torres
7,Miller PLC,miller plc
8,Jackson-Morgan,jackson morgan
9,"Blake, Mays and Anderson",blake mays and anderson


## 8) Feature Engineering (Time + Age)

We create foundational features used in:
- EDA (seasonality, weekday patterns)
- clustering (behavioral signatures)
- LLM summaries (human-readable insights)

Features:
- year, month, day, weekday
- age, age_group

In [59]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["weekday"] = df["date"].dt.day_name()

In [61]:
today = pd.Timestamp.today().normalize()
df["age"] = ((today - df["birthdate"]).dt.days // 365).astype("Int64")

# Optional: age grouping (adjust bins as desired)
df["age_group"] = pd.cut(
    df["age"].astype(float),
    bins=[0, 17, 25, 35, 45, 55, 65, 120],
    labels=["<18", "18-25", "26-35", "36-45", "46-55", "56-65", "65+"]
)

df[["birthdate", "age", "age_group"]].head(10)

,birthdate,age,age_group
0,2002-10-20,23,18-25
1,1985-10-24,40,36-45
2,1981-10-25,44,36-45
3,1977-10-26,48,46-55
4,1951-11-02,74,65+
5,2001-10-20,24,18-25
6,1976-10-26,49,46-55
7,1968-10-28,57,56-65
8,1957-10-31,68,65+
9,1974-10-27,51,46-55


## 9) Category Standardization

Even if categories are present, we standardize casing/whitespace so grouping is consistent.

In [64]:
df["category"] = df["category"].astype(str).str.strip().str.title()
df["category"].value_counts().head(15)

category
Restaurant     8413
Market         8382
Travel         8377
Electronics    8324
Clothing       8261
Cosmetic       8243
Name: count, dtype: int64

## 10) Create Customer-Level Behavior Table

In addition to saving cleaned transactions, we create a customer-level table
that will be used for clustering and persona creation.

This table contains:
- total spend
- average spend
- spending volatility (std)
- transaction volume
- diversity of spending categories
- demographic anchors (age, gender)

In [67]:
customer_features = df.groupby("customer_id").agg(
    total_spend=("transaction_amount", "sum"),
    avg_spend=("transaction_amount", "mean"),
    spend_std=("transaction_amount", "std"),
    transaction_count=("transaction_amount", "count"),
    unique_categories=("category", "nunique"),
    age=("age", "first"),
    gender=("gender", "first"),
)

customer_features["spend_std"] = customer_features["spend_std"].fillna(0)  # if only 1 txn
customer_features.head()

,total_spend,avg_spend,spend_std,transaction_count,unique_categories,age,gender
customer_id,,,,,,,
29,27.02,27.02,0.0,1,1,42,M
51,1898.56,1898.56,0.0,1,1,21,Unknown
54,166.30,166.30,0.0,1,1,44,M
83,125.85,125.85,0.0,1,1,74,Unknown
90,18.16,18.16,0.0,1,1,20,M


## 11) Optional: Category Mix Table (Customer x Category Spend)

This produces a matrix that helps:
- detect dominant spend areas per customer
- improve persona interpretability
- generate better LLM summaries

We store both:
1) raw category spend matrix
2) category spend proportions

In [70]:
cust_cat_spend = pd.pivot_table(
    df,
    index="customer_id",
    columns="category",
    values="transaction_amount",
    aggfunc="sum",
    fill_value=0
)

cust_cat_prop = cust_cat_spend.div(cust_cat_spend.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)

cust_cat_spend.head(), cust_cat_prop.head()

(category     Clothing  Cosmetic  Electronics  Market  Restaurant   Travel
 customer_id                                                              
 29               0.00       0.0          0.0     0.0       27.02     0.00
 51               0.00       0.0          0.0     0.0        0.00  1898.56
 54             166.30       0.0          0.0     0.0        0.00     0.00
 83             125.85       0.0          0.0     0.0        0.00     0.00
 90               0.00       0.0          0.0     0.0       18.16     0.00,
 category     Clothing  Cosmetic  Electronics  Market  Restaurant  Travel
 customer_id                                                             
 29                0.0       0.0          0.0     0.0         1.0     0.0
 51                0.0       0.0          0.0     0.0         0.0     1.0
 54                1.0       0.0          0.0     0.0         0.0     0.0
 83                1.0       0.0          0.0     0.0         0.0     0.0
 90                0.0       0

## 12) Final Quality Checks

Before saving outputs, we confirm:
- no missing required fields in the cleaned dataset
- dates are parsed
- transaction amounts are non-zero
- expected row counts look reasonable

In [73]:
checks = {
    "rows": len(df),
    "unique_customers": df["customer_id"].nunique(),
    "missing_date_pct": float(df["date"].isna().mean() * 100),
    "missing_birthdate_pct": float(df["birthdate"].isna().mean() * 100),
    "zero_amount_rows": int((df["transaction_amount"] == 0).sum()),
    "missing_category_pct": float(df["category"].isna().mean() * 100),
    "missing_merchant_pct": float(df["merchant_name"].isna().mean() * 100),
}
checks

{'rows': 50000,
 'unique_customers': 50000,
 'missing_date_pct': 0.0,
 'missing_birthdate_pct': 0.0,
 'zero_amount_rows': 0,
 'missing_category_pct': 0.0,
 'missing_merchant_pct': 0.0}

## 13) Persist Outputs

Deliverables from this notebook:
1. `cleaned_transactions.csv` — cleaned transaction-level dataset
2. `customer_behavior_features.csv` — customer-level behavioral features
3. `customer_category_spend.csv` — customer x category spend (optional but recommended)
4. `customer_category_proportions.csv` — category proportions (optional but recommended)

In [76]:
df.to_csv("cleaned_transactions.csv", index=False)
customer_features.to_csv("customer_behavior_features.csv", index=True)
cust_cat_spend.to_csv("customer_category_spend.csv", index=True)
cust_cat_prop.to_csv("customer_category_proportions.csv", index=True)

print("✅ Saved:")
print("- cleaned_transactions.csv")
print("- customer_behavior_features.csv")
print("- customer_category_spend.csv")
print("- customer_category_proportions.csv")

✅ Saved:
- cleaned_transactions.csv
- customer_behavior_features.csv
- customer_category_spend.csv
- customer_category_proportions.csv
